<a href="https://colab.research.google.com/github/thimarArda/BinX_AI_-_ML_Training/blob/main/Week7_CNNs_RNN_Transformers/Day%204%20%E2%80%94%20Attention%20%26%20Transformers/Banking_Chatbot_with_DistilBERT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Transformer Text Project - Intent Classification Chatbot

> - **Model:** DistilBERT
> - **Dataset:** Banking77
> - **Project direction:** Intent-based chatbot / semantic text understanding

Project Idea

```text
User message
     ↓
DistilBERT
     ↓
Intent
     ↓
Chatbot response
```

Example

```text
User:
"I want to know why my card payment was declined."

             ↓

DistilBERT

             ↓

Intent:
cash_withdrawal / card_payment_wrong_exchange_rate / card_payment_fee_charged
...
```

Dataser: Banking77 contains user queries belonging to 77 different banking-related intents

## Step 1 : Install the Libraries

In [1]:
!pip install -U transformers datasets==2.18.0 accelerate evaluate scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 510.5/510.5 kB 13.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 9.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 72.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.9/170.9 kB 18.0 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.3.0
    Uninstalling fsspec-2025.3.0:
      Successfully uninstalled fsspec-2025.3.0
  Attempting uninstall: scikit-learn
    Found existing installation: scikit-learn 1.6.1
    Uninstalling scikit-learn-1.6.1:
      Successfully uninstalled scikit-learn-1.6.1
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.

In [2]:
import numpy as np
import pandas as pd

from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)

from sklearn.metrics import accuracy_score, precision_recall_fscore_support

## Step 2 : Loading and Inspecting the Dataset
> this dataset is provided by HuggingFace

In [3]:
from datasets import load_dataset

dataset = load_dataset("PolyAI/banking77")

/usr/local/lib/python3.13/dist-packages/datasets/load.py:1461: FutureWarning: The repository for PolyAI/banking77 contains custom code which must be executed to correctly load the dataset. You can inspect the repository content at https://hf.co/datasets/PolyAI/banking77
You can avoid this message in future by passing the argument `trust_remote_code=True`.
Passing `trust_remote_code=True` will be mandatory to load this dataset from the next major release of `datasets`.
  warnings.warn(


/root/.cache/huggingface/modules/datasets_modules/datasets/PolyAI--banking77/17ffc2ed47c2ed928bee64127ff1dbc97204cb974c2f980becae7c864007aed9/banking77.py:25: SyntaxWarning: invalid escape sequence '\~'
  author      = {I{\~{n}}igo Casanueva and Tadas Temcinas and Daniela Gerz and Matthew Henderson and Ivan Vulic},


Generating train split:   0%|          | 0/10003 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/3080 [00:00<?, ? examples/s]

In [4]:
print(dataset)

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 10003
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 3080
    })
})


In [5]:
label_names = dataset["train"].features["label"].names

print(label_names)
print("Number of classes:", len(label_names))

print("Printing out one example" ,dataset["train"][0])

['activate_my_card', 'age_limit', 'apple_pay_or_google_pay', 'atm_support', 'automatic_top_up', 'balance_not_updated_after_bank_transfer', 'balance_not_updated_after_cheque_or_cash_deposit', 'beneficiary_not_allowed', 'cancel_transfer', 'card_about_to_expire', 'card_acceptance', 'card_arrival', 'card_delivery_estimate', 'card_linking', 'card_not_working', 'card_payment_fee_charged', 'card_payment_not_recognised', 'card_payment_wrong_exchange_rate', 'card_swallowed', 'cash_withdrawal_charge', 'cash_withdrawal_not_recognised', 'change_pin', 'compromised_card', 'contactless_not_working', 'country_support', 'declined_card_payment', 'declined_cash_withdrawal', 'declined_transfer', 'direct_debit_payment_not_recognised', 'disposable_card_limits', 'edit_personal_details', 'exchange_charge', 'exchange_rate', 'exchange_via_app', 'extra_charge_on_statement', 'failed_transfer', 'fiat_currency_support', 'get_disposable_virtual_card', 'get_physical_card', 'getting_spare_card', 'getting_virtual_card'

## Step 3 : Create a Validation Set
> Createing a validation split from the training data for model tuning

In [6]:
split_dataset = dataset["train"].train_test_split(
    test_size=0.1,
    seed=42
)

train_dataset = split_dataset["train"]
val_dataset = split_dataset["test"]

test_dataset = dataset["test"]

## Step 4 : Load the DistilBERT Tokenizer

> The tokenizer converts the text into numerical information that the Transformer can process.

In this  step, we  initialize the tokenizer associated with the `distilbert-base-uncased` pre-trained model and verify it's output using a simple test sentence.

- `AutoTokenizer.from_pretrained`: Downloads and loads the vocabulary file, vocabulary configuration, and tokenization rules specifically used by the distilbert-base-uncased model. Using AutoTokenizer ensures using the exact same preprocessing logic that DistilBERT was trained on.

- "uncased": Indicates that the tokenizer automatically converts all text to lowercase and removes accent marks before splitting words into tokens ("Password" becomes "password").

- tokenizer(text): Converts raw text strings into numerical tensors that neural networks can process.

In [7]:
model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)



config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

In [8]:
text = "I forgot my password"

tokens = tokenizer(text)

print(tokens)

{'input_ids': [101, 1045, 9471, 2026, 20786, 102], 'token_type_ids': [0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1]}


## Step 5 : Tokenize the Dataset

> This step scales up the tokenization process to transform the entire dataset (train, validation, and test splits) into uniform, structured batches ready to feed into the neural network during training.

In [9]:
def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        padding="max_length",
        truncation=True,
        max_length=128
    )

# Applying the tokenization
tokenized_train = train_dataset.map(
    tokenize_function,
    batched=True
)

tokenized_val = val_dataset.map(
    tokenize_function,
    batched=True
)

tokenized_test = test_dataset.map(
    tokenize_function,
    batched=True
)

Map:   0%|          | 0/9002 [00:00<?, ? examples/s]

Map:   0%|          | 0/1001 [00:00<?, ? examples/s]

Map:   0%|          | 0/3080 [00:00<?, ? examples/s]

## Step 6 : Load The Treansformer DistilBERT
> We take a pre-trained DistilBERT and adding a classification task on top.

In [10]:
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=len(label_names)
)

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


## Step 7 : Define Evaluation Metrics

> - Accuracy
> - Precision
> - Recall
> - F1-score

In [11]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred

    predictions = np.argmax(logits, axis=-1)

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels,
        predictions,
        average="weighted",
        zero_division=0
    )

    accuracy = accuracy_score(labels, predictions)

    return {
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1
    }

## Step 8 :  Configure Training


In this step, we set the **training settings for the DistilBERT model** using `TrainingArguments`. These settings tell the Trainer how we want the model to learn, such as the learning rate, batch size, number of training epochs, when to evaluate and save the model, and how to handle the best model.


In [12]:
training_args = TrainingArguments(
    output_dir="./distilbert-banking77",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    load_best_model_at_end=True,
    report_to="none"
)

## Step 9 : Create the Trainer and Train the Transformer

What happens during the trainng process:

```text
input text : "I forgot my card PIN"
          ↓
      Tokenizer
          ↓
    Token IDs
          ↓
      DistilBERT
          ↓
   Text representation
          ↓
 Classification layer
          ↓
  77 probabilities
          ↓
 Predicted intent
          ↓
Compare with true intent
          ↓
Calculate loss
          ↓
Backpropagation
          ↓
Update model
```

In [13]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    processing_class=tokenizer,
    compute_metrics=compute_metrics
)

trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,3.396574,2.121189,0.644356,0.696748,0.644356,0.601253
2,1.841183,1.212919,0.787213,0.806831,0.787213,0.765397
3,1.189417,0.997322,0.820180,0.842452,0.820180,0.807776


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1689, training_loss=2.0180135109214263, metrics={'train_runtime': 322.2219, 'train_samples_per_second': 83.812, 'train_steps_per_second': 5.242, 'total_flos': 895549856592384.0, 'train_loss': 2.0180135109214263, 'epoch': 3.0})

## Step 10 :  Evaluate the Transformer

In [14]:
results = trainer.evaluate(tokenized_test)
print(results)

Training Loss,Validation Loss,Epoch,Accuracy,Precision,Recall,F1
1.189417,1.058934,3,0.815260,0.824739,0.815260,0.797395


{'eval_loss': 1.0589336156845093, 'eval_accuracy': 0.8152597402597402, 'eval_precision': 0.8247385780593324, 'eval_recall': 0.8152597402597402, 'eval_f1': 0.7973947639839588}


## Step 11: Testing the Model on a provided text

> Here I added a cell to input a forgin text of my own creation to see how the model performs

In [25]:
text = "I forgot my PIN"

inputs = tokenizer(
    text,
    return_tensors="pt",
    truncation=True,
    padding=True
)

# Move the inputs to the same device as the model:
import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model.to(device)
inputs = {key: value.to(device) for key, value in inputs.items()}

# make prediction
with torch.no_grad():
    outputs = model(**inputs)

prediction = torch.argmax(outputs.logits, dim=-1).item()

print("Predicted intent:", label_names[prediction])

Predicted intent: get_physical_card


For the specific text "I forgot my PIN", the model predicted the intent "get_physical_card".
The model gave the text, "get_physical_card" intent which might not be the most intuitive intent. It's likely that a more accurate intent would be related to 'change_pin' or 'pin_blocked', if those intents exist in the dataset.


In [26]:
desired_intents = ['change_pin', 'pin_blocked', 'get_physical_card']

print("Checking for desired intents:")
for intent in desired_intents:
    if intent in label_names:
        print(f"- Intent '{intent}' exists in the dataset.")
    else:
        print(f"- Intent '{intent}' DOES NOT exist in the dataset.")

Checking for desired intents:
- Intent 'change_pin' exists in the dataset.
- Intent 'pin_blocked' exists in the dataset.
- Intent 'get_physical_card' exists in the dataset.


This code proved that these intents exist in the dataset, the model missed the prediction